# Load all AC sessions together

Load the complete neural data, features, and neuron IDs from all six AC collections into memory. Every session is retained at its original dtype and bin size. `neural_data[s]`, `features[s]`, and `neuron_ids[s]` refer to the same session. `session_info.iloc[s]` records its animal, headstage, and source session index. Sessions are ordered by filename, then source session index.

The measurements describe loading and retaining the data. Later analyses can require additional memory. Rerunning the loading cell first releases the previous loaded lists.

In [ ]:
from pathlib import Path
import gc
import pickle
import time

import pandas as pd
import psutil
from IPython.display import display

# Resolve the shared data folder from the workspace, repository, or notebooks folder.
data_dir = next(
    (parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Cannot locate the workspace data directory.")

dt = 0.005
expected_collections = {
    "FETA_hs_0_AC", "HERCULE_hs_0_AC",
    "MMELOIK_hs_0_AC", "MMELOIK_hs_1_AC",
    "NAPOLEON_hs_0_AC", "NAPOLEON_hs_1_AC",
}
prefixes = sorted(
    path.name.removesuffix("_data_ss")
    for path in data_dir.glob(f"*_AC_playback_{dt}_data_ss")
)
if {prefix.removesuffix(f"_playback_{dt}") for prefix in prefixes} != expected_collections:
    raise ValueError(f"Unexpected AC collections: {prefixes}")
paths = [data_dir / f"{prefix}_{suffix}" for prefix in prefixes
         for suffix in ("data_ss", "feature_ss", "neuron_idx_ss")]
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)

GiB = 1024 ** 3
process = psutil.Process()
memory = psutil.virtual_memory()
print(f"Data directory: {data_dir}")
print(f"AC collections: {len(prefixes)}; files: {len(paths)}")
print(f"Files on disk: {sum(path.stat().st_size for path in paths) / GiB:.2f} GiB")
print(f"Physical RAM: {memory.total / GiB:.2f} GiB; available: {memory.available / GiB:.2f} GiB")


In [ ]:
# Clear prior results before rerunning so they do not double the memory requirement.
neural_data = []
features = []
neuron_ids = []
session_rows = []
load_rows = []
session_info = None
load_summary = None
collection_neural = collection_features = collection_ids = None
gc.collect()
baseline_rss = process.memory_info().rss
started = time.perf_counter()

for prefix in prefixes:
    collection_started = time.perf_counter()
    print(f"Loading {prefix}...", flush=True)
    with (data_dir / f"{prefix}_data_ss").open("rb") as stream:
        collection_neural = pickle.load(stream)
    with (data_dir / f"{prefix}_feature_ss").open("rb") as stream:
        collection_features = pickle.load(stream)
    with (data_dir / f"{prefix}_neuron_idx_ss").open("rb") as stream:
        collection_ids = pickle.load(stream)

    if not (len(collection_neural) == len(collection_features) == len(collection_ids)):
        raise ValueError(f"Session counts do not match for {prefix}.")
    animal, _, headstage, region, *_ = prefix.split("_")
    for source_session in range(len(collection_neural)):
        shape = collection_neural[source_session].shape
        if shape != (len(collection_ids[source_session]), len(collection_features[source_session])):
            raise ValueError(f"Alignment failed for {prefix}, session {source_session}.")
        session_rows.append({
            "animal": animal, "headstage": int(headstage), "region": region,
            "source_session": source_session, "prefix": prefix,
            "n_units": shape[0], "n_bins": shape[1],
        })

    # Extend the lists with references to the loaded objects.
    neural_data.extend(collection_neural)
    features.extend(collection_features)
    neuron_ids.extend(collection_ids)
    collection_neural = collection_features = collection_ids = None
    memory = psutil.virtual_memory()
    info = process.memory_info()
    load_rows.append({
        "collection": prefix, "sessions_loaded_total": len(neural_data),
        "load_seconds": time.perf_counter() - collection_started,
        "process_rss_GiB": info.rss / GiB,
        "process_private_GiB": getattr(info, "private", info.rss) / GiB,
        "available_RAM_GiB": memory.available / GiB,
    })
    print(f"  {len(neural_data)} sessions retained; RSS {info.rss / GiB:.2f} GiB; "
          f"available RAM {memory.available / GiB:.2f} GiB", flush=True)

elapsed = time.perf_counter() - started
session_info = pd.DataFrame(session_rows).rename_axis("session")
load_summary = pd.DataFrame(load_rows)
print(f"Loaded all {len(neural_data)} AC session entries in {elapsed:.1f} seconds.")


In [ ]:
# Read the shape and one scalar from every retained session after loading finishes.
for s in range(len(neural_data)):
    if neural_data[s].size:
        first_count = neural_data[s].flat[0]
    if len(features[s]):
        first_condition = features[s]["Condition"].iloc[0]
    if neuron_ids[s]:
        first_neuron_id = neuron_ids[s][0]

neural_GiB = sum(session.nbytes for session in neural_data) / GiB
feature_GiB = sum(session.memory_usage(index=True, deep=True).sum() for session in features) / GiB
info = process.memory_info()
print(f"All {len(neural_data)} retained session entries are accessible and aligned.")
print(f"Neural array storage: {neural_GiB:.2f} GiB")
print(f"Feature table storage: {feature_GiB:.2f} GiB")
print(f"Process resident RAM: {info.rss / GiB:.2f} GiB (baseline {baseline_rss / GiB:.2f} GiB)")
print(f"Process private memory: {getattr(info, 'private', info.rss) / GiB:.2f} GiB")
print(f"Peak process resident RAM: {getattr(info, 'peak_wset', info.rss) / GiB:.2f} GiB")
print(f"System RAM available: {psutil.virtual_memory().available / GiB:.2f} GiB")
print("Session entries count each headstage separately.")
display(load_summary)
display(session_info.groupby(["animal", "headstage", "region"]).size().rename("sessions").to_frame())


Use `session_info` to select sessions for analysis, then access `neural_data[s]`, `features[s]["Condition"]`, and `neuron_ids[s]` directly. The feature `Condition` distinguishes tracking (0) and playback (1).

## Result from 16 November 2026

All 80 AC session entries loaded together and passed alignment checks in 24.2 seconds. The neural arrays occupied 6.54 GiB and feature tables 5.44 GiB. The process used 12.28 GiB of private memory, with 5.59 GiB resident in physical RAM at the final measurement. The PC had 15.39 GiB of physical RAM and 1.39 GiB available before loading. The memory measurements indicate substantial paging under this workload. Loading succeeded, while the speed and memory needs of further analyses remain to be measured.

These saved outputs came from executing the notebook cells with the shared `.venv` interpreter. The test process has exited and released its loaded data. Run the notebook to retain the data in your interactive kernel.